# Phase 3 — Classical Machine Learning

I explore feature extraction and participant-independent baseline model selection.

> **Research integrity:** Figures labeled synthetic are not clinical EEG observations. Run code locally in VS Code using the project `.venv` interpreter.

## 1. Feature engineering

Each channel contributes time-domain statistics, Hjorth parameters, and normalized spectral band power. These are intentionally simple baselines.

In [ ]:
from pathlib import Path
import numpy as np
from eeg_sleep.synthetic import make_demo_dataset
from eeg_sleep.dataset import load_arrays
from eeg_sleep.features import extract_features, feature_names
p=Path("data/processed/synthetic_demo.npz")
if not p.exists():make_demo_dataset(p,n_subjects=10,repeats_per_stage=2)
X,y,groups,ids,fs=load_arrays(p)
F=extract_features(X,fs=fs)
print("EEG:",X.shape,"features:",F.shape)
print(feature_names(X.shape[1]))

## 2. Subject-wise data separation

Do not split individual 30-second epochs at random. Such leakage would place data from the same participant in both training and testing.

In [ ]:
from eeg_sleep.splits import subject_split
tr,va,te=subject_split(groups)
print("Groups:",{"train":sorted(set(groups[tr])),"val":sorted(set(groups[va])),"test":sorted(set(groups[te]))})

## 3. Train and compare candidates

This optional cell performs actual **synthetic software** model training. The function chooses an estimator by validation macro F1; only afterward is the held-out subject test evaluated.

In [ ]:
RUN_SYNTHETIC_BASELINES = False  # Set True when you want to exercise the trainer
if RUN_SYNTHETIC_BASELINES:
    from eeg_sleep.ml import train_classical
    metrics=train_classical(p,"results/ml")
    print("Software test macro F1 (not biomedical evidence):", metrics["macro_f1"])
else:
    print("Training intentionally disabled; enable after studying the split.")

## 4. Inspect recorded outcomes if training was run

In [ ]:
import json
path=Path("results/ml/ml_metrics.json")
if path.exists():
    metrics=json.loads(path.read_text())
    print("Chosen baseline:", metrics["selected_by_validation"])
    print("Validation metrics:", metrics["validation_macro_f1"])
else: print("No saved metrics yet.")

## Exercises

1. How is Hjorth mobility defined? 2. Why fit the scaler on training subjects only? 3. Why is macro F1 often preferable to accuracy with imbalanced stages? 4. Why is model selection on test data invalid?